# EXPERT_REVIEW table
Written by: Capleton

This notebook makes a CSV file that fits the table expert_review in our PostgreSQL database.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [1]:
import os
import pandas as pd

## STEP 2: read the files
We read the cleaned expert reviews and the movie table. The movie table is needed because every review must get the movie_id of its movie. Before we change anything, we look at the first rows and count the empty values, so we know what goes into the database. We also check that movie has a unique movie_id, otherwise a review could be linked to two movies.

In [ ]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\clean"

expert = pd.read_csv(os.path.join(folder, "ExpertReviews_clean.csv"))
movie = pd.read_csv(os.path.join(folder, "movie.csv"))

print("Expert reviews:", len(expert))
print("Movies:", len(movie))

print(expert[["url", "idvscore", "posemo", "negemo"]].head())
print(expert[["url", "idvscore", "posemo", "negemo"]].isnull().sum())

print("Check, movie has movie_id and url:", "movie_id" in movie.columns and "url" in movie.columns)
print("Check, movie_id is unique:", movie["movie_id"].is_unique)

Expert reviews: 238973
Movies: 11332
                                        url  idvscore  posemo  negemo
0  https://www.metacritic.com/movie/bronson     100.0    0.00    0.00
1  https://www.metacritic.com/movie/bronson      90.0    6.67    0.00
2  https://www.metacritic.com/movie/bronson      90.0    0.00    2.50
3  https://www.metacritic.com/movie/bronson      83.0    5.13    5.13
4  https://www.metacritic.com/movie/bronson      80.0    0.00    0.00
url         0
idvscore    2
posemo      0
negemo      0
dtype: int64
Check, movie has movie_id and url: True
Check, movie_id is unique: True


## STEP 3: keep only the ERD columns and give them the ERD names
The review file has around 90 columns, but our ERD only uses the columns that answer our research questions: the score and the positive and negative emotion. The names must be the same as in the table in PostgreSQL, otherwise pgAdmin does not know which value goes into which column.

In [ ]:
expert = expert[["url", "review_score", "pos_emotion_pct", "neg_emotion_pct"]]
expert = expert.rename(columns={"idvscore": "review_score",  "posemo": "pos_emotion_pct", "negemo": "neg_emotion_pct"})

print(list(expert.columns))

print("Check, columns are correct:", list(expert.columns) == ["url", "review_score", "pos_emotion_pct", "neg_emotion_pct"])

['url', 'review_score', 'pos_emotion_pct', 'neg_emotion_pct']
Check, columns are correct: True


## STEP 4: fill movie_id with the url
movie_id is the foreign key to MOVIE. The expert reviews and the movie table come from the same Metacritic data, so they share the url. The url is unique per movie, so we do not need a match key here: we look up the url in MOVIE and take the movie_id. Spaces at the start and end of the url are removed first, so the same url is written the same way in both files. The check makes sure the number of reviews stays the same: if it grows, a url was found twice in movie.

In [11]:
expert_before = len(expert)
expert["url"] = expert["url"].str.strip()
movie["url"] = movie["url"].str.strip()

expert = expert.merge(movie[["movie_id", "url"]], on="url", how="left")

print("Expert reviews with a movie:", expert["movie_id"].notna().sum())
print("Expert reviews without a movie:", expert["movie_id"].isna().sum())
print("Check, no review got two movies:", len(expert) == expert_before)

Expert reviews with a movie: 238592
Expert reviews without a movie: 381
Check, no review got two movies: True


## STEP 5: remove reviews without a movie
movie_id is NOT NULL in expert_review, and a foreign key must point to a movie that exists in MOVIE. A review of a movie that is not in our database cannot be imported, so it is removed. Int64 keeps movie_id a whole number, because PostgreSQL does not accept 12.0 in an INT column.

In [12]:
expert = expert.dropna(subset=["movie_id"])
expert["movie_id"] = expert["movie_id"].astype("Int64")

print("Expert reviews left:", len(expert))
print("Check, no empty movie_id left:", expert["movie_id"].isna().sum() == 0)

Expert reviews left: 238592
Check, no empty movie_id left: True


## STEP 6: give every review an expert_review_id
expert_review_id is the primary key. The database needs it to tell every review apart, so it must be unique and can never be empty. We make it in Python, the same as all other IDs in our database.

In [17]:
expert = expert.reset_index(drop=True)
expert["expert_review_id"] = expert.index + 1

print("Check, expert_review_id is unique:", expert["expert_review_id"].is_unique)
print(expert.head())
print(movie.head())

Check, expert_review_id is unique: True
                                        url  review_score  pos_emotion_pct  \
0  https://www.metacritic.com/movie/bronson         100.0             0.00   
1  https://www.metacritic.com/movie/bronson          90.0             6.67   
2  https://www.metacritic.com/movie/bronson          90.0             0.00   
3  https://www.metacritic.com/movie/bronson          83.0             5.13   
4  https://www.metacritic.com/movie/bronson          80.0             0.00   

   neg_emotion_pct  movie_id  expert_review_id  
0             0.00      1565                 1  
1             0.00      1565                 2  
2             2.50      1565                 3  
3             5.13      1565                 4  
4             0.00      1565                 5  
   movie_id  director_id  actor_id  \
0         1         3790       NaN   
1         2         1379       NaN   
2         3          803       NaN   
3         4         2634       NaN   
4      

## STEP 7: put the columns in the same order as the table
The order of the columns in the CSV must be the same as in the table in PostgreSQL: first the primary key, then the foreign key, then the rest.

In [18]:
expert = expert[["expert_review_id", "movie_id", "url", "review_score",
 "pos_emotion_pct", "neg_emotion_pct"]]

print(expert.head())
print("Check, primary key is first:", expert.columns[0] == "expert_review_id")

   expert_review_id  movie_id                                       url  \
0                 1      1565  https://www.metacritic.com/movie/bronson   
1                 2      1565  https://www.metacritic.com/movie/bronson   
2                 3      1565  https://www.metacritic.com/movie/bronson   
3                 4      1565  https://www.metacritic.com/movie/bronson   
4                 5      1565  https://www.metacritic.com/movie/bronson   

   review_score  pos_emotion_pct  neg_emotion_pct  
0         100.0             0.00             0.00  
1          90.0             6.67             0.00  
2          90.0             0.00             2.50  
3          83.0             5.13             5.13  
4          80.0             0.00             0.00  
Check, primary key is first: True


## STEP 8: save as CSV
This file is imported into expert_review in pgAdmin. index=False prevents an extra column with row numbers, because that column does not exist in the table. At the end we read the file back, to check that everything was saved.

In [19]:
clean_folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\clean"
new_file = os.path.join(clean_folder, "expert_review_clean.csv")

expert.to_csv(new_file, index=False)

print("Saved:", new_file)


check = pd.read_csv(new_file)
print("Check, same number of rows in the file:", len(check) == len(expert))

Saved: C:\Users\caple\OneDrive - HvA\Master\Data management\clean\expert_review_clean.csv
Check, same number of rows in the file: True
